# Phase 2 — Literature Review

**Course:** AML-DL Project | **Authors:** Prerak Arya (230039), Saikiran Bompelliwar (230046)

This notebook presents a comprehensive literature review covering the theoretical evolution and practical application of anomaly detection methods for network intrusion detection. We trace the development from classical statistical approaches through kernel methods to modern deep generative models, identifying the specific gaps that motivate our Phase 2 model choices.

---

## 1. Introduction to Anomaly-Based Intrusion Detection

Network Intrusion Detection Systems (NIDS) are broadly classified into two categories: **signature-based** and **anomaly-based** detection. Signature-based systems maintain a database of known attack patterns and match incoming traffic against these signatures. While effective for known threats, they are fundamentally incapable of detecting zero-day attacks — novel exploits that have not been previously catalogued (Buczak & Guven, 2016).

Anomaly-based detection takes a complementary approach: instead of defining what attacks look like, it defines what *normal* traffic looks like and flags deviations. This paradigm shift — from supervised classification to **semi-supervised anomaly detection** — is particularly valuable in cybersecurity, where:

1. **Attack diversity is unbounded:** New attack vectors emerge continuously, making exhaustive labelling impossible
2. **Normal traffic is abundant:** Clean baseline data is readily available in most networks
3. **The cost of false negatives is asymmetric:** Missing a real intrusion is far more costly than a false alarm
4. **Attack distributions shift:** The statistical properties of attacks change over time (concept drift)

The NSL-KDD dataset (Tavallaee et al., 2009) addresses limitations of the original KDD Cup 99 dataset by removing redundant records and rebalancing the difficulty distribution. It contains 41 features spanning four categories: basic TCP/IP features (duration, protocol, bytes), content features (failed logins, root access), time-based traffic features (connection rates), and host-based traffic features (same-service rates). Attacks are categorised into four families:

| Family | Description | Example Attacks | Detection Difficulty |
|--------|-------------|-----------------|---------------------|
| **DoS** | Deny service availability | Neptune, Smurf, Back | Moderate — high volume |
| **Probe** | Surveillance/scanning | Portsweep, Satan, Nmap | Moderate — distinctive patterns |
| **R2L** | Remote-to-local unauthorised access | Guess_passwd, Warezmaster, Phf | High — mimics normal behaviour |
| **U2R** | User-to-root privilege escalation | Buffer_overflow, Rootkit, Perl | Very high — subtle indicators |

Our Phase 1 analysis confirmed that R2L and U2R attacks exhibit near-normal feature distributions, motivating the need for methods that can learn non-linear feature interactions beyond raw feature-level thresholds.

---

## 2. Statistical and Proximity-Based Methods

### 2.1 Foundational Statistical Approaches

Denning (1987) introduced the first formal model for anomaly-based intrusion detection, proposing that user behaviour could be profiled through statistical measures (means, variances, frequency counts) and that significant deviations from these profiles indicate intrusions. This model established the foundational assumption that persists across the field: **normal behaviour is statistically regular; anomalous behaviour is not**.

The statistical approach was later formalised through various parametric and non-parametric methods. Z-Score thresholding — which our Phase 1 implements — assumes Gaussian feature distributions and flags samples beyond $k$ standard deviations. While computationally efficient ($O(nd)$ for $n$ samples, $d$ features), this assumption is violated by many network traffic features that exhibit heavy-tailed, multi-modal, or zero-inflated distributions (Chandola et al., 2009).

Our Phase 1 Z-Score baseline achieved 85.27% accuracy and 89.93% AUC, demonstrating the ceiling of parametric approaches on NSL-KDD.

### 2.2 Isolation Forest and Tree-Based Methods

Liu et al. (2008) introduced Isolation Forest, which takes a fundamentally different approach: instead of profiling normal behaviour, it directly isolates anomalies. The key insight is that anomalies, being "few and different", are easier to separate from the rest of the data — they require fewer random partitions to isolate.

Each isolation tree randomly selects a feature and a split value between the feature's minimum and maximum. Anomalies, which tend to have extreme values or unusual feature combinations, are isolated near the root of the tree (short path length), while normal samples require many splits and fall deeper. The anomaly score is based on the average path length across an ensemble of trees:

$$s(x, n) = 2^{-\frac{E[h(x)]}{c(n)}}$$

where $h(x)$ is the path length and $c(n)$ is a normalisation factor.

Our Phase 1 Isolation Forest achieved 84.86% accuracy but a substantially higher AUC of 93.91%, suggesting it provides better ranking of anomalous vs. normal samples despite similar classification accuracy. The gap between accuracy and AUC is characteristic of threshold sensitivity — Isolation Forest's strength lies in its anomaly ranking rather than its binary decision at a fixed contamination rate.

### 2.3 Limitations Motivating Advanced Methods

Both statistical and proximity-based methods operate on **raw features** — they cannot discover higher-order feature interactions that may be necessary to detect sophisticated attacks. For example, a R2L attack may have individually normal values for `src_bytes`, `dst_bytes`, and `service`, but the *combination* (low bytes, specific service, specific flag pattern) may be anomalous. Capturing these interactions requires either:
- **Kernel methods** that implicitly operate in high-dimensional feature spaces (Section 3)
- **Deep learning methods** that explicitly learn non-linear feature representations (Sections 4-5)

---

## 3. Kernel Methods: One-Class SVM

### 3.1 Theoretical Foundation

Schölkopf et al. (2001) extended the Support Vector Machine framework to the one-class (novelty detection) setting. Rather than finding a hyperplane that separates two classes, One-Class SVM (OC-SVM) finds a hyperplane in a kernel-induced feature space that maximally separates the training data from the origin:

$$\min_{w, \rho, \xi} \frac{1}{2}\|w\|^2 + \frac{1}{\nu n}\sum_{i=1}^n \xi_i - \rho$$

The parameter $\nu \in (0, 1]$ has a dual interpretation: it is an **upper bound on the fraction of outliers** in the training set and a **lower bound on the fraction of support vectors**. This makes $\nu$ directly interpretable in terms of the expected anomaly rate.

The RBF (Radial Basis Function) kernel $K(x_i, x_j) = \exp(-\gamma \|x_i - x_j\|^2)$ maps data into an infinite-dimensional feature space, enabling complex non-linear decision boundaries without explicitly computing the feature map. The bandwidth parameter $\gamma$ controls the smoothness of the boundary — small $\gamma$ produces smoother boundaries (more generalisation), while large $\gamma$ produces tighter boundaries (more overfitting).

### 3.2 Application to Network Intrusion Detection

Tax & Duin (2004) demonstrated that OC-SVM with appropriate kernel selection can achieve competitive performance on network intrusion datasets when trained exclusively on normal traffic. Key findings from the literature:

- **RBF kernel is preferred** for network intrusion detection due to the heterogeneous nature of network features — continuous rates, discrete counts, and binary indicators coexist (Eskin et al., 2002)
- **Computational cost scales as $O(n^2)$** for kernel matrix computation and $O(n^3)$ for the QP solver, limiting applicability to large datasets without subsampling
- **Feature scaling is critical** — without standardisation, features with large magnitudes (e.g., `src_bytes`) dominate the distance computation in the RBF kernel

Choi et al. (2021) applied OC-SVM specifically to NSL-KDD in a semi-supervised setting, confirming that training on normal traffic only (our approach) produces competitive results. Their work motivates our implementation in Phase 2 as an advanced ML baseline between Phase 1's statistical methods and our deep learning models.

### 3.3 Subsampling Strategy

The quadratic memory requirement of the kernel matrix ($O(n^2)$) makes full-dataset training impractical on our 8GB M2 MacBook. With ~54,000 normal training samples, the full kernel matrix would require $\sim 54000^2 \times 8 \approx 23$ GB. We therefore subsample 15,000 normal records — reducing memory to $\sim 1.8$ GB while retaining representative coverage through random sampling.

---

## 4. Autoencoders for Anomaly Detection

### 4.1 The Reconstruction-Based Paradigm

Autoencoders learn to compress and reconstruct their input through an encoder-decoder bottleneck architecture. When trained exclusively on normal data, the model learns a compressed representation of the normal data manifold. At inference time, anomalies — data points that lie off this learned manifold — produce higher reconstruction errors, providing a natural anomaly score.

Sakurada & Yairi (2014) formalised this approach, demonstrating that autoencoders provide a non-linear generalisation of Principal Component Analysis (PCA). While PCA finds linear subspaces that capture maximum variance, autoencoders learn non-linear manifolds through their activation functions — crucial for capturing the complex feature interactions in network traffic data.

### 4.2 Autoencoder Variants for Intrusion Detection

Song et al. (2021) conducted a comprehensive comparison of autoencoder variants on NSL-KDD:

| Variant | Key Feature | Advantage | Limitation |
|---------|-------------|-----------|------------|
| **Vanilla AE** | Standard bottleneck | Simple, fast | Information loss at bottleneck |
| **Sparse AE** | L1 penalty on activations | Feature selection | Hard to tune sparsity coefficient |
| **Denoising AE** | Corrupted input | Robustness | Noise level is a hyperparameter |
| **Contractive AE** | Jacobian penalty | Smooth representations | Expensive to compute |
| **VAE** | Probabilistic latent space | Uncertainty quantification | Can prioritise regularity over reconstruction |

Their findings show that architectural choice significantly impacts detection performance, with no single variant dominating across all attack types. This motivates our approach of implementing complementary architectures (deterministic AE with skip connections and probabilistic VAE) rather than a single model.

### 4.3 Skip Connections: Cross-Domain Adaptation from ResNet

He et al. (2016) introduced residual learning through skip connections to solve the degradation problem in very deep image classifiers. The key insight is that learning residual mappings $F(x) = H(x) - x$ is easier than learning the direct mapping $H(x)$, because the identity mapping $x$ provides a shortcut for gradient flow.

We adapt this insight from the **computer vision domain** to **tabular anomaly detection** — a cross-domain architectural transfer. In our autoencoder, skip connections serve a different but related purpose:

1. **Gradient flow:** Direct connections from encoder to decoder layers prevent vanishing gradients in the deep bottleneck architecture
2. **Information preservation:** Fine-grained feature details bypass the lossy bottleneck compression, enabling detection of subtle anomalies that would otherwise be lost
3. **Dual representation:** The bottleneck learns abstract normal patterns for generalisation, while skip connections preserve specific feature values for discrimination

This architectural innovation is not standard in the anomaly detection literature — most autoencoder-based IDS use vanilla architectures without residual connections (Pang et al., 2021). Our implementation tests whether the benefits observed in deep image networks transfer to tabular anomaly detection.

### 4.4 Weighted Reconstruction Loss

Standard MSE loss weights all features equally, but not all features are equally informative for anomaly detection. Features with low variance in normal data (e.g., `num_root`, `su_attempted`) carry high information when they deviate — a deviation in a typically-constant feature is more suspicious than a deviation in a high-variance feature.

We implement inverse-variance weighting: $w_j = 1 / (\text{Var}(x_j) + \epsilon)$, giving higher weight to low-variance features. This emphasises the features that our Phase 1 analysis identified as most discriminative.

---

## 5. Variational Autoencoders and Probabilistic Scoring

### 5.1 From Deterministic to Probabilistic Representations

While standard autoencoders provide point estimates of the latent representation, Variational Autoencoders (VAEs) (Kingma & Welling, 2014) model the latent space as a probability distribution. This distinction is crucial for anomaly detection:

- **Deterministic AE:** $z = f_\text{enc}(x)$ → single latent vector → single reconstruction error
- **VAE:** $z \sim q(z|x) = \mathcal{N}(\mu(x), \sigma^2(x))$ → distribution of latent vectors → distribution of reconstruction errors

The VAE framework is derived from variational inference. The goal is to maximise the marginal likelihood $p(x)$, but this requires integrating over all possible latent codes $z$ — intractable for deep models. The Evidence Lower Bound (ELBO) provides a tractable surrogate:

$$\text{ELBO} = \underbrace{\mathbb{E}_{q(z|x)}[\log p(x|z)]}_{\text{reconstruction quality}} - \underbrace{D_{\text{KL}}(q(z|x) \| p(z))}_{\text{latent regularity}}$$

The first term encourages accurate reconstruction; the second regularises the latent space toward the prior $p(z) = \mathcal{N}(0, I)$.

### 5.2 The $\beta$-VAE Framework

Higgins et al. (2017) introduced a tunable coefficient $\beta$ on the KL divergence term:

$$\mathcal{L}_{\beta\text{-VAE}} = \mathcal{L}_\text{recon} + \beta \cdot D_\text{KL}$$

For anomaly detection, the reconstruction term directly determines detection quality — the better the model reconstructs normal data, the more distinct anomalies become. Setting $\beta < 1$ (we use $\beta = 0.5$) prioritises reconstruction fidelity over latent space regularity. This is the opposite of the original $\beta$-VAE motivation (disentangled representations), but is well-justified for our task: we need precise reconstruction, not interpretable latent factors.

### 5.3 Reconstruction Probability Scoring

An & Cho (2015) proposed using **reconstruction probability** rather than deterministic reconstruction error as the anomaly score. Instead of encoding a sample once and measuring reconstruction error, they sample $L$ latent vectors from the posterior and average the reconstruction errors:

$$\text{Score}(x) = \frac{1}{L}\sum_{l=1}^L \|x - D(z^{(l)})\|^2, \quad z^{(l)} \sim q(z|x)$$

This Monte Carlo estimate provides several advantages:
1. **Robustness:** Averaging over multiple samples reduces sensitivity to individual latent draws
2. **Uncertainty incorporation:** Samples with high latent variance (uncertain encodings) produce diverse reconstructions, naturally increasing the anomaly score
3. **Calibration:** The score reflects the model's confidence in the reconstruction, not just the point estimate

We use $L = 50$ Monte Carlo samples, balancing computational cost with estimation quality.

### 5.4 The Reparameterisation Trick

Training VAEs requires backpropagating through the stochastic sampling operation $z \sim q(z|x)$. The reparameterisation trick (Kingma & Welling, 2014) makes this possible by expressing the random variable as a deterministic transformation of a noise variable:

$$z = \mu + \sigma \odot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

This reformulation moves the stochasticity outside the computational graph, enabling standard gradient computation through $\mu$ and $\sigma$. Without this trick, training VAEs with gradient descent would be impossible.

---

## 6. Curriculum Learning for Training Stability

Bengio et al. (2009) demonstrated that training models on data ordered from "easy" to "hard" — mimicking human learning — can improve both training stability and final performance. The key insight is that easy samples provide a robust initial signal that prevents the model from getting stuck in poor local optima during early training.

For our anomaly detection task, we define sample difficulty using **L2 distance from the centroid** of the normal training data. Samples close to the centroid represent "typical" normal traffic (easy), while those far from the centroid are edge cases that may share characteristics with anomalous traffic (hard).

Our three-stage curriculum:
1. **Easy (33%):** Most typical normal traffic — establishes the core normal manifold
2. **Medium (66%):** Extends to moderately unusual traffic — refines the boundary
3. **Hard (100%):** Includes all normal traffic, including edge cases — sharpens the boundary

This progressive approach is particularly valuable for semi-supervised anomaly detection, where the model must learn a tight boundary around normal data without any anomalous examples for contrast.

---

## 7. Gap Analysis and Our Contributions

### 7.1 Identified Gaps in Prior Work

| Gap | Evidence | Our Solution |
|-----|----------|--------------|
| **Information loss in bottleneck** | Vanilla AE discards fine-grained features that distinguish subtle attacks | Skip connections preserve features while forcing abstract bottleneck learning |
| **Fixed reconstruction loss** | Equal feature weighting ignores discriminative importance | Inverse-variance weighted MSE emphasises low-variance, high-information features |
| **Training instability** | Deep AEs on heterogeneous tabular data often converge to poor optima | Curriculum learning from easy to hard samples provides stable training signal |
| **Point estimate scoring** | Single reconstruction error is sensitive to latent sampling noise | Monte Carlo reconstruction probability (50 samples) provides robust scoring |
| **Over-regularised VAE** | Standard $\beta=1$ sacrifices reconstruction for latent regularity | $\beta=0.5$ prioritises detection-critical reconstruction fidelity |

### 7.2 Cross-Domain Innovation

Our primary architectural innovation — applying skip connections from deep image classifiers (ResNet, He et al., 2016) to tabular anomaly detection — represents a **cross-domain adaptation**. While skip connections are ubiquitous in computer vision and NLP (Transformer residual connections), their application to tabular autoencoders for intrusion detection is uncommon in the literature. Our ablation study in the training notebooks will quantify the specific benefit.

---

## 8. References

1. An, J. & Cho, S. (2015). Variational autoencoder based anomaly detection using reconstruction probability. *SNU Data Mining Center Technical Report*.
2. Bengio, Y., Louradour, J., Collobert, R. & Weston, J. (2009). Curriculum learning. *ICML*, 41–48.
3. Buczak, A. L. & Guven, E. (2016). A survey of data mining and machine learning methods for cyber security intrusion detection. *IEEE Communications Surveys & Tutorials*, 18(2), 1153–1176.
4. Chandola, V., Banerjee, A. & Kumar, V. (2009). Anomaly detection: A survey. *ACM Computing Surveys*, 41(3), 1–58.
5. Choi, H. et al. (2021). Application of deep autoencoder as a one-class classifier for unsupervised network intrusion detection. *PeerJ Computer Science*, 7, e420.
6. Denning, D. E. (1987). An intrusion-detection model. *IEEE Transactions on Software Engineering*, SE-13(2), 222–232.
7. Eskin, E., Arnold, A., Prerau, M., Portnoy, L. & Stolfo, S. (2002). A geometric framework for unsupervised anomaly detection. *Applications of Data Mining in Computer Security*, Springer.
8. He, K., Zhang, X., Ren, S. & Sun, J. (2016). Deep residual learning for image recognition. *CVPR*, 770–778.
9. Higgins, I. et al. (2017). $\beta$-VAE: Learning basic visual concepts with a constrained variational framework. *ICLR*.
10. Kingma, D. P. & Welling, M. (2014). Auto-encoding variational Bayes. *ICLR*.
11. Liu, F. T., Ting, K. M. & Zhou, Z.-H. (2008). Isolation forest. *ICDM*, 413–422.
12. Malhotra, P. et al. (2016). LSTM-based encoder-decoder for multi-sensor anomaly detection. *ICML Workshop on Anomaly Detection*.
13. Pang, G., Shen, C., Cao, L. & van den Hengel, A. (2021). Deep learning for anomaly detection: A review. *ACM Computing Surveys*, 54(2), Article 38.
14. Sakurada, M. & Yairi, T. (2014). Anomaly detection using autoencoders with nonlinear dimensionality reduction. *MLSDA Workshop*, ACM.
15. Schölkopf, B., Platt, J. C., Shawe-Taylor, J., Smola, A. J. & Williamson, R. C. (2001). Estimating the support of a high-dimensional distribution. *Neural Computation*, 13(7), 1443–1471.
16. Song, Y., Hyun, S. & Cheong, Y.-G. (2021). Analysis of autoencoders for network intrusion detection. *Sensors*, 21(13), 4294.
17. Tavallaee, M., Bagheri, E., Lu, W. & Ghorbani, A. A. (2009). A detailed analysis of the KDD CUP 99 data set. *IEEE Symposium on Computational Intelligence for Security and Defense Applications (CISDA)*.
18. Tax, D. M. J. & Duin, R. P. W. (2004). Support vector data description. *Machine Learning*, 54(1), 45–66.
19. Tishby, N., Pereira, F. C. & Bialek, W. (2000). The information bottleneck method. *arXiv:physics/0004057*.
20. Yin, C. et al. (2017). A deep learning approach for intrusion detection using recurrent neural networks. *IEEE Access*, 5, 21954–21961.

---

*This literature review will be referenced throughout the subsequent notebooks. Each model notebook cites specific papers relevant to its implementation choices.*